# HumanAI Detect - On Isleme (Asama 2, Colab GPU)

**Amac:** `human`, `ai_raw`, `ai_humanized` siniflari icin Asama 2 on islemeyi
(Stanza dilbilimsel analiz + BERTurk pseudo-perplexity + burstiness) Colab T4 GPU
uzerinde calistirmak. Yerel CPU'da tek ornek ~200 saniye surdugu icin (maskeli-LM
perplexity yontemi) bu adim GPU gerektiriyor.

**Veri:** Uc sinif da 3000'er ornekle tam (9000 toplam). Olculen hiz ~11 sn/ornek
(GPU'da) -> 9000 ornek icin tahmini toplam **~27.5 saat**. Bu TEK bir Colab oturumunda
bitmez (free tier session/idle limitlerini asar). Script her ornegi aninda diske
yazdigi icin (checkpoint) birden fazla oturuma bolunerek calistirilmali.

**ONEMLI -- ai_raw/ai_humanized YENIDEN URETILIYOR (2026-07-04), bu notebooku
calistirmadan ONCE `colab_data_collection.ipynb` ile yeni ai_raw/ai_humanized
uretilmis ve yerele indirilmis olmali.** Onceki uretim ortalama ~234 kelime/ornek
cikmisti (insan ~615 kelime/ornek) -- bu uzunluk farki tek basina siniflari %71
macro-F1 ile ayirt edebiliyordu (ciddi confound). Yeni uretimde ornekler insan
dagilimina yakin (~605 kelime ort.) hedefleniyor.

**KRITIK -- resume/checkpoint tuzagi:** Asama 2 checkpoint'i ornekleri ID'ye gore
esler (orn. `ai_raw_transformers_0000`). Yeni ai_raw/ai_humanized uretimi AYNI ID
semasini kullanir ama icerik tamamen farkli (kisa yerine uzun metin). Bu yuzden
4b. hucrede **SADECE `human` interim dosyasini** geri yukle (insan verisi
degismedi, gecerli) -- **`ai_raw`/`ai_humanized` interim dosyalarini KESINLIKLE
yukleme**, yuklersen script eski kisa-metin sonuclarini "zaten islenmis" sanip
atlar ve confound'u sessizce geri getirir. ai_raw/ai_humanized SIFIRDAN islenmeli.

**Onceki oturumdan kalan, SADECE human icin gecerli (2026-07-03):** 500 human
ornegi zaten islenmis durumda (yerelde `data/interim/human.jsonl`). Bu dosyayi
4b. hucreyle Colab'a geri yuklersen script bunu checkpoint'ten tanir ve TEKRAR
islemez.

**Adimlar:**
1. GPU kontrolu
2. Repo'yu klonla, bagimliliklari kur
3. Ham veriyi (3000'er ornek, ai_raw/ai_humanized YENI/uzun surumler) yerel
   makineden yukle (`human.jsonl`, `ai_raw.jsonl`, `ai_humanized.jsonl`)
4. (Opsiyonel) SADECE onceki oturumdan kalan `human` interim sonucunu yukle (resume)
5. `preprocess.py --limit 3000` calistir (her sinif icin tum 3000 ornek, kaldigi yerden devam eder)
6. Sonuclari dogrudan yerel makineye indir (oturum sonunda veya bitince)

**Onkosul:** Yukleme hucresi (3.) calistirildiginda acilan dosya sec penceresinden
su 3 dosyayi (tek seferde, hepsini birden secerek) yukle:
- `data/raw/human/human.jsonl` (3000 kayit)
- `data/raw/ai_raw/ai_raw.jsonl` (3000 kayit, YENI/uzun surum)
- `data/raw/ai_humanized/ai_humanized.jsonl` (3000 kayit, YENI/uzun surum)

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 3a. Repo klonla (GitHub URL'ini kendi repo'nla degistir)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect

In [ ]:
# 3b. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install bitsandbytes accelerate -q

In [ ]:
# 3b-2. Stanza Turkce model indir (pip install'dan sonra AYRI hucrede calistir,
# ayni hucrede yapinca Colab yeni kurulan paketi bazen taniyamiyor)
import stanza
stanza.download('tr')

In [ ]:
# 3c. .env olustur (bu adim icin API key gerekmez)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')

In [ ]:
# 4. Ham veriyi yerel makineden dogrudan yukle
# Acilan pencereden human.jsonl, ai_raw.jsonl, ai_humanized.jsonl dosyalarini
# (hangi isimle olursa olsun, adlarindan taninir) hep birlikte sec.
from google.colab import files
from pathlib import Path

uploaded = files.upload()

# Sira onemli: 'ai_humanized' icinde 'human' gectigi icin once ozel etiketler,
# en sonda genel 'human' kontrol edilir.
for fname in uploaded:
    for label in ['ai_humanized', 'ai_raw', 'human']:
        if label in fname:
            dst = Path(f'data/raw/{label}/{label}.jsonl')
            dst.parent.mkdir(parents=True, exist_ok=True)
            Path(fname).replace(dst)
            lines = sum(1 for _ in open(dst, encoding='utf-8'))
            print(f'{label}: {lines} kayit -> {dst}')
            break
    else:
        print(f'!!! taninmayan dosya: {fname} (adinda human/ai_raw/ai_humanized gecmiyor)')

In [ ]:
# 4b. (Opsiyonel) SADECE `human` interim sonucunu geri yukle -- insan verisi
# degismedi, checkpoint'ten guvenle taninir. ai_raw/ai_humanized YUKLEME (yukaridaki
# ONEMLI notuna bak): o ikisi yeni/uzun metinle SIFIRDAN islenmeli, eski checkpoint
# kullanilirsa eski kisa-metin sonuclari sessizce karisir.
# Acilan pencereden SADECE human.jsonl dosyasini sec (varsa).
# ILK kez calistiriyorsan (hic interim dosyan yoksa) bu hucreyi CALISTIRMA.
from google.colab import files
from pathlib import Path

uploaded = files.upload()

for fname in uploaded:
    if 'human' in fname and 'ai_humanized' not in fname:
        dst = Path('data/interim/human/human.jsonl')
        dst.parent.mkdir(parents=True, exist_ok=True)
        Path(fname).replace(dst)
        lines = sum(1 for _ in open(dst, encoding='utf-8'))
        print(f'human: {lines} onceden islenmis ornek geri yuklendi -> {dst}')
    else:
        print(f'!!! {fname} YUKLENMEDI -- sadece human.jsonl kabul edilir (ai_raw/ai_humanized icin SIFIRDAN islenmeli)')

In [ ]:
# 5. On isleme (her sinif icin tum 3000 ornek) -- GPU'da calisir, checkpoint'li
# (~11 sn/ornek, toplam 9000 ornek icin ~27.5 saat -- birden fazla oturuma bolunmeli)
!python scripts/preprocess.py --label all --limit 3000

In [ ]:
# 6. Sonuclari dogrudan yerel makineye indir (oturum sonunda veya tamamen bitince
# calistir; henuz bitmemis olsa bile o ana kadarki kismi sonuclari indirip
# bir sonraki oturumda 4b. hucreyle geri yukleyerek devam edebilirsin)
from google.colab import files
from pathlib import Path

for label in ['human', 'ai_raw', 'ai_humanized']:
    src = Path(f'data/interim/{label}/{label}.jsonl')
    if src.exists():
        lines = sum(1 for _ in open(src, encoding='utf-8'))
        print(f'{label}: {lines} ornek -> indiriliyor')
        files.download(str(src))
    else:
        print(f'{label}: dosya bulunamadi!')

## Yerel Makineye Aktarim

6. hucre calisinca tarayici 3 dosyayi (human.jsonl, ai_raw.jsonl, ai_humanized.jsonl)
tek tek indirir (indirilenler genelde Indirilenler/Downloads klasorune duser). Bunlari
her seferinde yerel projede suraya koy (mevcut dosyalarin uzerine yaz):
   - `data/interim/human/human.jsonl`
   - `data/interim/ai_raw/ai_raw.jsonl`
   - `data/interim/ai_humanized/ai_humanized.jsonl`

**9000 ornek TEK oturumda bitmeyecegi icin (~27.5 saat tahmini):**
1. Oturumu actigin kadar calistir (Colab baglantiyi kesene/idle timeout'a kadar).
2. 6. hucreyi calistirip o ana kadarki kismi sonuclari indir, yukaridaki gibi yerele koy.
3. Yeni bir Colab oturumu ac, 1-3. hucreleri calistir (GPU/klon/kurulum), SONRA
   4b. hucreyle bu indirdigin kismi interim dosyalarini geri yukle, sonra 5. hucreyi
   tekrar calistir -- script kaldigi yerden (checkpoint) devam eder.
4. Tum 9000 ornek islenene kadar 1-3 adimlarini tekrarla.

Tum 9000 ornek tamamlaninca `build_features.py` ve `train_model.py --balance`
calistirilarak gercek 9000'lik performans olculebilir.